# 03 — Model Experiments

Training and evaluation of all 4 models on the complete 4-year dataset.

**Models:** Ridge Regression, Random Forest, XGBoost, LSTM  
**Data:** 107,208 hourly observations (Aug 2022 – Aug 2026)  
**Targets:** AQI 24h, 48h, 72h ahead

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

from src.config import load_environment
load_environment()

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data from Hopsworks

In [ ]:
from src.feature_store import get_feature_store
store = get_feature_store()
df = store.get_features('aqi_features_prod', version=1)
print(f'Loaded {len(df):,} rows from Hopsworks')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Cities: {df["location_id"].unique().tolist()}')

## 2. Prepare Features and Targets

In [ ]:
from src.features.feature_engineering import add_lag_features, add_rolling_features, add_time_features
from src.utils.epa_aqi import calculate_pm10_aqi, calculate_pm25_aqi

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df = df.sort_values('timestamp').reset_index(drop=True)
df = add_time_features(df)

if 'aqi' not in df.columns or df['aqi'].isna().all():
    df['pm25_aqi'] = df['pm25'].apply(lambda x: calculate_pm25_aqi(x) if pd.notna(x) else None)
    df['pm10_aqi'] = df['pm10'].apply(lambda x: calculate_pm10_aqi(x) if pd.notna(x) else None)
    df['aqi'] = df[['pm25_aqi', 'pm10_aqi']].max(axis=1)

df = add_lag_features(df)
df = add_rolling_features(df)

for lag in [1, 6, 12, 24, 48, 72]:
    df[f'aqi_lag_{lag}h'] = df.groupby('location_id')['aqi'].shift(lag)
for lag in [1, 6, 12, 24, 48, 72]:
    df[f'pm25_lag_{lag}h'] = df.groupby('location_id')['pm25'].shift(lag)

target_cols = ['target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h']
has_targets = all(c in df.columns and not df[c].isna().all() for c in target_cols)
if not has_targets:
    for h, c in [(24, 'target_aqi_24h'), (48, 'target_aqi_48h'), (72, 'target_aqi_72h')]:
        df[c] = df.groupby('location_id')['aqi'].shift(-h)

df = df.dropna(subset=target_cols)
print(f'Usable rows: {len(df):,}')

exclude = {'timestamp', 'location_id', 'city_name', 'data_source', 'collected_at',
           'is_training_valid', 'target_aqi_24h', 'target_aqi_48h', 'target_aqi_72h',
           'us_aqi', 'us_aqi_pm25', 'us_aqi_pm10', 'pm25_aqi', 'pm10_aqi', 'aqi_category'}
string_cols = set(df.select_dtypes(include=['object']).columns)
feature_cols = [c for c in df.columns if c not in exclude and c not in string_cols]
print(f'Features: {len(feature_cols)}')

X = df[feature_cols].values
y = df[target_cols].values
X = np.nan_to_num(X, nan=0.0)

n = len(X)
train_end = int(n * 0.72)
val_end = int(n * 0.80)
X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]
print(f'Train: {len(X_train):,}, Val: {len(X_val):,}, Test: {len(X_test):,}')

## 3. Train All Models

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

def evaluate(model, X, y):
    y_pred = model.predict(X)
    horizons = ['24h', '48h', '72h']
    results = {
        'mae': float(mean_absolute_error(y, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y, y_pred))),
        'r2': float(r2_score(y, y_pred)),
    }
    for i, h in enumerate(horizons):
        results[f'mae_{h}'] = float(mean_absolute_error(y[:, i], y_pred[:, i]))
        results[f'rmse_{h}'] = float(np.sqrt(mean_squared_error(y[:, i], y_pred[:, i])))
        results[f'r2_{h}'] = float(r2_score(y[:, i], y_pred[:, i]))
    return results

models = {}

# Ridge
print('Training Ridge...')
t0 = time.time()
ridge = MultiOutputRegressor(Ridge(alpha=1.0))
ridge.fit(X_train, y_train)
models['Ridge'] = {'model': ridge, 'time': time.time() - t0}
print(f'  Done ({models["Ridge"]["time"]:.1f}s)')

# Random Forest
print('Training Random Forest...')
t0 = time.time()
rf = MultiOutputRegressor(RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1))
rf.fit(X_train, y_train)
models['RandomForest'] = {'model': rf, 'time': time.time() - t0}
print(f'  Done ({models["RandomForest"]["time"]:.1f}s)')

# XGBoost
print('Training XGBoost...')
t0 = time.time()
xgb_model = MultiOutputRegressor(xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                                   subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1))
xgb_model.fit(X_train, y_train)
models['XGBoost'] = {'model': xgb_model, 'time': time.time() - t0}
print(f'  Done ({models["XGBoost"]["time"]:.1f}s)')

# LSTM
print('Training LSTM...')
t0 = time.time()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

X_tr_lstm = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_v_lstm = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
X_te_lstm = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

lstm = Sequential([
    LSTM(64, input_shape=(1, X_train.shape[1]), return_sequences=True),
    Dropout(0.2), LSTM(32), Dropout(0.2),
    Dense(16, activation='relu'), Dense(3),
])
lstm.compile(optimizer='adam', loss='mse', metrics=['mae'])
lstm.fit(X_tr_lstm, y_train, validation_data=(X_v_lstm, y_val),
         epochs=50, batch_size=32, callbacks=[EarlyStopping(patience=5, restore_best_weights=True)], verbose=0)

class LSTMWrapper:
    def __init__(self, m): self.model = m
    def predict(self, X):
        if X.ndim == 2: X = X.reshape((X.shape[0], 1, X.shape[1]))
        return self.model.predict(X, verbose=0)

models['LSTM'] = {'model': LSTMWrapper(lstm), 'time': time.time() - t0}
print(f'  Done ({models["LSTM"]["time"]:.1f}s)')

## 4. Evaluate All Models

In [ ]:
results = {}
for name, data in models.items():
    val = evaluate(data['model'], X_val, y_val)
    test = evaluate(data['model'], X_test, y_test)
    results[name] = {'val': val, 'test': test, 'time': data['time']}
    print(f'{name}: Val MAE={val["mae"]:.2f}, Test MAE={test["mae"]:.2f}, Test R2={test["r2"]:.4f}')

## 5. Comparison Table

In [ ]:
comparison = []
for name in ['Ridge', 'RandomForest', 'XGBoost', 'LSTM']:
    r = results[name]
    comparison.append({
        'Model': name,
        'Val MAE': f'{r["val"]["mae"]:.2f}',
        'Val R²': f'{r["val"]["r2"]:.4f}',
        'Test MAE': f'{r["test"]["mae"]:.2f}',
        'Test RMSE': f'{r["test"]["rmse"]:.2f}',
        'Test R²': f'{r["test"]["r2"]:.4f}',
        'Train Time': f'{r["time"]:.1f}s',
    })
pd.DataFrame(comparison)

## 6. Per-Horizon Comparison

In [ ]:
horizons = ['24h', '48h', '72h']
horizon_comparison = []
for name in ['Ridge', 'RandomForest', 'XGBoost', 'LSTM']:
    r = results[name]['test']
    for h in horizons:
        horizon_comparison.append({
            'Model': name,
            'Horizon': h,
            'MAE': r[f'mae_{h}'],
            'RMSE': r[f'rmse_{h}'],
            'R²': r[f'r2_{h}'],
        })

hdf = pd.DataFrame(horizon_comparison)
print(hdf.to_string(index=False))

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
model_names = ['Ridge', 'RandomForest', 'XGBoost', 'LSTM']
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

for ax, h in zip(axes, horizons):
    h_mae = [results[m]['test'][f'mae_{h}'] for m in model_names]
    bars = ax.bar(model_names, h_mae, color=colors)
    ax.set_title(f'{h} Horizon — Test MAE')
    ax.set_ylabel('MAE')
    # Highlight best
    best_idx = np.argmin(h_mae)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)

plt.suptitle('Per-Horizon Model Comparison (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Model Selection

In [ ]:
# Find best model by test MAE
test_maes = {name: results[name]['test']['mae'] for name in model_names}
best_model = min(test_maes, key=test_maes.get)
print(f'Best model by Test MAE: {best_model} (MAE={test_maes[best_model]:.2f})')
print(f'\nTest MAE comparison:')
for name, mae in sorted(test_maes.items(), key=lambda x: x[1]):
    print(f'  {name}: {mae:.2f}')